In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Extracion de datos
movies_df = spark.read.parquet(f"{silver_folder_path}/movies")
production_countries_df = spark.read.parquet(f"{silver_folder_path}/production_countries")
countries_df = spark.read.parquet(f"{silver_folder_path}/countries")

movies_df.show(4)
production_countries_df.show(4)
countries_df.show(4)

In [0]:
#Tabla de movies filtrada con los campos y datos que necesitamos

movies_df = movies_df.filter(
                               (col("year_release_date") >= 2015)
                             )\
                    .select(movies_df.movie_id,
                            movies_df.year_release_date, 
                            movies_df.budget,
                            movies_df.revenue
                            )
movies_df.show(4)

In [0]:
#eneramos la tabla agregada con los campos solicitados
country_production_df = countries_df.join( production_countries_df,
                                           countries_df.country_id == production_countries_df.country_id,
                                          "inner"
                               )\
                          .select(production_countries_df.movie_id, countries_df.country_name)


movies_country_df = movies_df.join(country_production_df,
                                    movies_df.movie_id == country_production_df.movie_id,
                                    "inner"
                                       )\
                              .select(movies_df["*"], country_production_df.country_name)


In [0]:
movies_country_df.show(4)

In [0]:

movies_country_agg_df = movies_country_df.groupBy("year_release_date", "country_name")\
                                         .agg(
                                              sum("budget").alias("sum_budget"),
                                              sum("revenue").alias("sum_revenue")   
                                            )

movies_country_agg_df.show(3)                                                    

In [0]:
#"movies_genre_agg_df.select("year_release_date")
result_group_movie_country_df = movies_country_agg_df.select( "year_release_date","country_name", "sum_budget", "sum_revenue")\
                                                     .withColumn("dense_rank", dense_rank().over(Window.partitionBy("year_release_date")
                                                                                                       .orderBy(desc("sum_budget"))
                                                                                                       .orderBy(desc("sum_revenue"))
                                                                                                 )
                                                                )
                                        
result_group_movie_country_df.display()

In [0]:

result_group_movie_country_df = add_ingestion_date(result_group_movie_country_df)
result_group_movie_country_df = add_env(result_group_movie_country_df)

In [0]:
#Guardamos en la capa gold 
result_group_movie_country_df.write.mode("overwrite").format("parquet").save(f"{gold_folder_path}/result_group_movie_country_df")

df = spark.read.parquet(f"{gold_folder_path}/result_group_movie_country_df")
display(df)
